### Simulate a New Incoming Batch (testing tool, not part of production pipeline)

This notebook is a manual testing tool used only to verify that the SCD Type 2 logic in `01_silver_pipeline` behaves correctly. It is never included in the scheduled job — in production, new rows would arrive in bronze from a real source (for example, an Auto Loader ingestion job).

* We take the most recent existing row for one sample video (`sample_id`) from the bronze table.
* We increase its `views` count, to simulate that the video gained more views since the last run.
* We update `trending_date` to today's date and `_ingestion_timestamp` to the current time, so this row is correctly recognized as the newest one during deduplication in Step 1.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql.functions import col, current_date, current_timestamp, date_format, to_date

sample_id = "-0CMnp02rNY"

df_bronze_new_row = (
    spark.table(source_bronze_table)
    .filter(col(business_key) == sample_id)
    .orderBy(col("trending_date").desc(), col("_ingestion_timestamp").desc())
    .limit(1)
    .withColumn("views", col("views") + 15000)
    .withColumn("trending_date", date_format(current_date(), "yy.dd.MM"))
    .withColumn("_ingestion_timestamp", current_timestamp())
)

(
    df_bronze_new_row.write
    .format("delta")
    .mode("append")
    .saveAsTable(source_bronze_table)
)

In [0]:
display(
    spark.table(source_bronze_table)
    .filter(col(business_key) == "-0CMnp02rNY")
    .select("video_id", "views", "trending_date")
    .orderBy(col("trending_date").desc())
)